# Hair 논문 기반 실험: MedSigLIP-448 Frozen Linear Probe

현재 배포 후보 `EfficientNet-B1·384·Label Smoothing 0.05·Adam v2`는 그대로 보존합니다.
이번 노트북은 Google의 의료 이미지·텍스트 사전학습 모델 **MedSigLIP-448**이 Hair의 두피
5개 클래스를 더 잘 분리하는지 Validation에서만 확인합니다. Test는 후보가 정해진 뒤 별도
노트북에서 한 번만 평가합니다.

- 공식 모델 카드: https://huggingface.co/google/medsiglip-448
- 공식 시작 안내: https://developers.google.com/health-ai-developer-foundations/medsiglip/get-started


## 1. 한 가지 연구 가설과 고정 조건

**가설:** 일반 ImageNet 특징보다 의료 영상으로 사전학습된 MedSigLIP 특징이 두피의 홍반,
각질, 비듬, 탈모와 피지 차이를 더 잘 분리한다.

고정 조건은 동일 clean ZIP, 클래스 순서, Augmented Train 15,047장, Original Validation
1,252장, seed 42입니다. 변경 변수는 특징 추출기뿐입니다. MedSigLIP 본체는 학습하지 않고
고정한 뒤 분류용 Linear 층 하나만 50 epoch 학습합니다. 이 방식은 전체 미세조정보다 계산량과
과적합 위험이 작아 첫 선별 실험에 적합합니다. MedSigLIP의 피부과 사전학습이 두피 현미경
영상에도 유효한지는 알려져 있지 않으므로 이 Validation 결과로 따로 판단합니다.


In [ ]:
DOMAIN = 'hair'
PROJECT_ROOT = '/content/drive/MyDrive/mediflow_Project'
DATA_ZIP = '/content/drive/MyDrive/mediflow_Project/datasets/hair_datasets.zip'
AUDIT_DIR = ''
EXPECTED_DATA_SHA256 = '2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156'
RESUME_DIR = ''
MODE = 'hair_medsiglip_linear'
SEED = 42
SEEDS = [42]
BATCH_SIZE = 8
LINEAR_EPOCHS = 50
LINEAR_BATCH_SIZE = 256
LINEAR_LEARNING_RATE = 1e-3
LINEAR_WEIGHT_DECAY = 1e-4
EMBEDDING_SHARD_SIZE = 128
MODEL_ID = 'google/medsiglip-448'
TRAIN_VARIANT = 'augmented'
RUN_EXPERIMENTS = ['hair_medsiglip_448_frozen_linear_seed_42']


## 2. Drive·GPU·패키지 확인

Colab 런타임은 **A100 GPU 권장**, L4도 사용할 수 있습니다. 128장 단위 embedding 캐시는
Drive 결과 폴더에 저장되므로 연결이 끊기면 같은 폴더를 `RESUME_DIR`에 넣어 이어갑니다.
모델 이용약관 동의와 Colab Secrets의 `HF_TOKEN` 읽기 권한이 필요합니다.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import zipfile

drive.mount('/content/drive', force_remount=True)
data_path = Path(DATA_ZIP)
if not data_path.is_file():
    available = sorted(path.name for path in (Path(PROJECT_ROOT) / 'datasets').iterdir())
    raise FileNotFoundError(f'데이터 ZIP을 찾을 수 없습니다: {data_path}\n확인된 항목: {available}')
if not zipfile.is_zipfile(data_path):
    raise ValueError(f'ZIP 형식이 아닙니다: {data_path}')

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN or not HF_TOKEN.startswith('hf_'):
    raise ValueError('Colab 왼쪽 열쇠(Secrets)에 HF_TOKEN을 저장하고 노트북 접근을 켜세요.')

%pip -q install tensorflow==2.20.0 keras==3.13.2 pandas matplotlib pillow tqdm   "transformers==4.53.2" "huggingface_hub>=0.33,<1" "accelerate>=1.8,<2" safetensors

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # 이 노트북의 GPU는 PyTorch MedSigLIP 전용

import torch
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형에서 GPU를 선택하세요.')
print('데이터:', data_path)
print('GPU:', torch.cuda.get_device_name(0))


## 3. 내장 재현 코드

별도 저장소 연결은 필요 없습니다. 실행 코드와 생성 당시 commit, 설정, 환경 버전이 결과 폴더에
저장됩니다. 이 셀은 수정하지 않습니다.


In [ ]:
import sys, types
SOURCES = {'common_engine': '"""Sequential domain-configured experiments; validation selection precedes any test inference.\n\nThe Colab notebook embeds an exact copy of this module so no repository checkout\nis needed in Colab. Completed trials are reused only after artifact verification.\nInterrupted attempts are preserved and restarted, not resumed mid-epoch.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\n\nPROTOCOL = "mediflow_common_v1"\nTRIALS = [\n    {"id": "b0_224_ce", "backbone": "B0", "size": 224, "loss": "ce"},\n    {"id": "b0_256_ce", "backbone": "B0", "size": 256, "loss": "ce"},\n    {"id": "b0_256_ls005", "backbone": "B0", "size": 256, "loss": "ls005"},\n    {"id": "b0_256_focal15", "backbone": "B0", "size": 256, "loss": "focal15"},\n    {"id": "b1_256_ls005", "backbone": "B1", "size": 256, "loss": "ls005"},\n]\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(".json-" + uuid.uuid4().hex[:12] + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef classification_metrics(truth, probabilities, count=5):\n    truth = np.asarray(truth, dtype=np.int64)\n    probabilities = np.asarray(probabilities)\n    if (\n        truth.ndim != 1\n        or not len(truth)\n        or probabilities.shape != (len(truth), count)\n        or not np.isfinite(probabilities).all()\n        or np.any(truth < 0)\n        or np.any(truth >= count)\n    ):\n        raise ValueError("Invalid evaluation arrays")\n    predictions = probabilities.argmax(axis=1)\n    cm = np.bincount(count * truth + predictions, minlength=count * count).reshape(count, count)\n    tp = np.diag(cm).astype(float)\n    precision = np.divide(tp, cm.sum(0), out=np.zeros(count), where=cm.sum(0) != 0)\n    recall = np.divide(tp, cm.sum(1), out=np.zeros(count), where=cm.sum(1) != 0)\n    f1 = np.divide(\n        2 * precision * recall,\n        precision + recall,\n        out=np.zeros(count),\n        where=precision + recall != 0,\n    )\n    return {\n        "accuracy": float(np.mean(truth == predictions)),\n        "macro_f1": float(f1.mean()),\n        "class_f1": f1.tolist(),\n        "precision": precision.tolist(),\n        "recall": recall.tolist(),\n        "support": cm.sum(1).tolist(),\n        "confusion_matrix": cm.tolist(),\n        "count": len(truth),\n    }\n\n\ndef predict_dataset(model, dataset):\n    truth, probabilities = [], []\n    for images, labels in dataset:\n        probabilities.extend(model(images, training=False).numpy())\n        truth.extend(np.argmax(labels.numpy(), axis=1))\n    return np.asarray(truth, dtype=np.int64), np.asarray(probabilities)\n\n\ndef save_predictions(path, paths, truth, probabilities):\n    if len(paths) != len(truth):\n        raise ValueError("File order and prediction count differ")\n    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "path",\n                "true_index",\n                "pred_index",\n                *[f"prob_C{i}" for i in range(probabilities.shape[1])],\n            ]\n        )\n        for name, target, probs in zip(paths, truth, probabilities, strict=True):\n            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])\n\n\ndef loss_function(name):\n    if name == "ce":\n        return keras.losses.CategoricalCrossentropy()\n    if name == "ls005":\n        return keras.losses.CategoricalCrossentropy(label_smoothing=0.05)\n    if name == "focal15":\n        return keras.losses.CategoricalFocalCrossentropy(alpha=1.0, gamma=1.5)\n    raise ValueError(name)\n\n\ndef build_model(spec):\n    builder = {\n        "B0": keras.applications.EfficientNetB0,\n        "B1": keras.applications.EfficientNetB1,\n        "V2S": keras.applications.EfficientNetV2S,\n    }\n    size = spec["size"]\n    backbone = builder[spec["backbone"]](\n        include_top=False, weights="imagenet", input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = backbone(inputs, training=False)\n    features = keras.layers.GlobalAveragePooling2D()(features)\n    features = keras.layers.Dropout(0.3)(features)\n    outputs = keras.layers.Dense(spec["class_count"], activation="softmax")(features)\n    model = keras.Model(inputs, outputs)\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-4),\n        loss=loss_function(spec["loss"]),\n        metrics=["accuracy"],\n    )\n    return model\n\n\ndef configure_partial(model, loss_name):\n    backbones = [\n        layer\n        for layer in model.layers\n        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()\n    ]\n    if len(backbones) != 1:\n        raise ValueError("Expected one EfficientNet backbone")\n    backbone = backbones[0]\n    backbone.trainable = True\n    for index, layer in enumerate(backbone.layers):\n        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-5), loss=loss_function(loss_name), metrics=["accuracy"]\n    )\n    return [layer.name for layer in backbone.layers if layer.trainable]\n\n\nclass HistoryBackup(keras.callbacks.Callback):\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self.values = {}\n\n    def on_epoch_end(self, epoch, logs=None):\n        for key, value in (logs or {}).items():\n            self.values.setdefault(key, []).append(float(value))\n        write_json(self.path, self.values)\n\n\ndef fit_stage(model, train, val, directory, name, epochs):\n    best = directory / (name + "_best.keras")\n    callbacks = [\n        keras.callbacks.ModelCheckpoint(\n            str(best), monitor="val_accuracy", mode="max", save_best_only=True\n        ),\n        keras.callbacks.CSVLogger(str(directory / (name + "_log.csv"))),\n        HistoryBackup(directory / (name + "_history.json")),\n        keras.callbacks.TerminateOnNaN(),\n    ]\n    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2)\n    values = {key: [float(v) for v in seq] for key, seq in history.history.items()}\n    if len(values.get("val_accuracy", [])) != epochs or not all(\n        np.isfinite(seq).all() for seq in values.values()\n    ):\n        raise RuntimeError("Incomplete or non-finite training; attempt retained")\n    model.save(directory / (name + "_last.keras"))\n    return values, best\n\n\ndef checkpoint_choice(baseline_score, new_score):\n    """Keep the earlier/simpler checkpoint on ties."""\n    return new_score > baseline_score\n\n\ndef cached_record(root, trial_id, signature):\n    trial_dir = Path(root) / trial_id\n    marker = trial_dir / "completed.json"\n    if not marker.exists():\n        return None\n    record = read_json(marker)\n    if record["signature"] != signature:\n        raise ValueError("Resume settings differ; use a new suite directory")\n    for relative, digest in record["artifact_hashes"].items():\n        target = (trial_dir / relative).resolve()\n        if not target.is_relative_to(trial_dir.resolve()) or file_hash(target) != digest:\n            raise ValueError("Completed artifact changed or corrupted: " + relative)\n    return record\n\n\ndef evaluate_to_files(model, dataset, paths, directory, prefix):\n    truth, probabilities = predict_dataset(model, dataset)\n    metrics = classification_metrics(truth, probabilities, probabilities.shape[1])\n    write_json(directory / (prefix + "_metrics.json"), metrics)\n    save_predictions(directory / (prefix + "_predictions.csv"), paths, truth, probabilities)\n    return metrics\n\n\ndef run_trial(spec, dataset_factory, root, signature, seed=42, epochs1=15, epochs2=10):\n    cached = cached_record(root, spec["id"], signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / spec["id"] / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    write_json(directory / "spec.json", spec)\n    train, _ = dataset_factory("train", spec["size"], True)\n    val, paths = dataset_factory("val", spec["size"], False)\n    started = time.monotonic()\n    model = build_model(spec)\n    h1, best1 = fit_stage(model, train, val, directory, "stage1", epochs1)\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    trainable = []\n    h2 = {key: [] for key in h1}\n    best2 = best1\n    if epochs2:\n        trainable = configure_partial(model, spec["loss"])\n        h2, best2 = fit_stage(model, train, val, directory, "stage2", epochs2)\n    del model\n    selected_stage = (\n        "stage2"\n        if checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"], default=-1.0))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": spec["id"],\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]) if h2["val_accuracy"] else None,\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef finish_record(directory, record):\n    write_json(directory / "record.json", record)\n    record["artifact_hashes"] = {\n        str(path.relative_to(directory.parent)): file_hash(path)\n        for path in directory.iterdir()\n        if path.is_file()\n    }\n    write_json(directory.parent / "completed.json", record)\n\n\ndef selected_model_path(root, record):\n    return Path(root) / record["id"] / record["attempt"] / record["selected_model"]\n\n\ndef extend_b1(parent, dataset_factory, root, signature, seed=42, epochs=5):\n    trial_id = "b1_256_ls005_extend5"\n    cached = cached_record(root, trial_id, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    parent_dir = Path(root) / parent["id"] / parent["attempt"]\n    # Continue from epoch 10\'s LAST checkpoint, including optimizer state.\n    source = parent_dir / "stage2_last.keras"\n    model = keras.models.load_model(source)\n    if model.optimizer is None:\n        raise ValueError("Extension requires saved optimizer")\n    train, _ = dataset_factory("train", 256, True)\n    val, paths = dataset_factory("val", 256, False)\n    started = time.monotonic()\n    history, best = fit_stage(model, train, val, directory, "extension", epochs)\n    del model\n    parent_best = selected_model_path(root, parent)\n    parent_score = max(parent["stage1_best_val"], parent["stage2_best_val"])\n    keep_extension = checkpoint_choice(parent_score, max(history["val_accuracy"]))\n    selected = directory / "selected.keras"\n    import shutil\n\n    shutil.copyfile(best if keep_extension else parent_best, selected)\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": trial_id,\n        "spec": {**parent["spec"], "id": trial_id},\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": "extension" if keep_extension else "parent_" + parent["selected_stage"],\n        "validation": metrics,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "training_seconds": time.monotonic() - started,\n        "history": {key: parent["history"][key] + history[key] for key in history},\n        "stage_boundary": parent["stage_boundary"],\n        "extension_boundary": len(parent["history"]["accuracy"]),\n        "parent_last_sha256": file_hash(source),\n        "optimizer_restored": True,\n        "extension_best_val": max(history["val_accuracy"]),\n        "stage1_best_val": parent["stage1_best_val"],\n        "stage2_best_val": parent["stage2_best_val"],\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef select_winner(records):\n    # Stable order preserves earlier experiments on exact ties.\n    return max(records, key=lambda record: record["validation"]["accuracy"])\n', 'common_workflow': '"""Standalone Colab orchestration; shared contracts, audit gates and artifacts."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import common_engine as engine\n\nEXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}\n\n\ndef signature(value):\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, ensure_ascii=False).encode()\n    ).hexdigest()\n\n\ndef extract_zip(source, destination):\n    """Validate every member before creating any output; no overwrite."""\n    destination = Path(destination).resolve()\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(source) as archive:\n        seen = set()\n        for item in archive.infolist():\n            name = item.orig_filename\n            path = PurePosixPath(name)\n            if (\n                path.is_absolute()\n                or ".." in path.parts\n                or "\\\\" in name\n                or ":" in name\n                or stat.S_ISLNK(item.external_attr >> 16)\n            ):\n                raise ValueError("Unsafe ZIP member: " + name)\n            key = name.rstrip("/").casefold()\n            if key in seen:\n                raise ValueError("Duplicate ZIP destination: " + name)\n            seen.add(key)\n        destination.mkdir(parents=True)\n        archive.extractall(destination)\n\n\ndef roots_for(extracted):\n    roots = {}\n    for kind in ("original", "augmented"):\n        matches = [\n            p\n            for p in Path(extracted).rglob("*")\n            if p.is_dir()\n            and p.name.lower() == kind\n            and all((p / s).is_dir() for s in ("train", "val", "test"))\n        ]\n        if len(matches) != 1:\n            raise ValueError(f"{kind}/train,val,test 구조를 하나로 확인하세요: {matches}")\n        roots[kind] = matches[0]\n    return roots\n\n\ndef verify_audit(directory, domain, classes, digest):\n    directory = Path(directory)\n    manifest = engine.read_json(directory / "audit_manifest.json")\n    for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv"):\n        if engine.file_hash(directory / name) != manifest[name]:\n            raise ValueError("검증 보고서가 변경됐습니다: " + name)\n    audit = engine.read_json(directory / "audit_summary.json")\n    if (\n        audit["domain"] != domain\n        or audit["class_names"] != classes\n        or audit["data_sha256"] != digest\n        or audit.get("protocol") != "common_audit_v1"\n        or audit["status"] != "mechanical_checks_passed_with_limitations"\n    ):\n        raise ValueError("대상/클래스/데이터가 다르거나 검증 문제가 있습니다. 공통 ①을 확인하세요.")\n    return audit\n\n\ndef prepare(config, profiles, sources, commit, local_parent="/content"):\n    c = dict(config)\n    c.setdefault("expected_data_sha256", "")\n    c.setdefault("seeds", [c["seed"]])\n    if c["domain"] not in profiles or c["mode"] not in (\n        "audit",\n        "comparison",\n        "suite",\n        "baseline3",\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "final_candidate",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n        "web_skin_pmg_final",\n        "web_skin_medsiglip_linear",\n        "hair_medsiglip_linear",\n    ):\n        raise ValueError("DOMAIN/MODE 설정을 확인하세요.")\n    if c["train_variant"] not in ("original", "augmented"):\n        raise ValueError("TRAIN_VARIANT는 original 또는 augmented입니다.")\n    for key in ("batch_size", "epochs1", "epochs2", "extension_epochs"):\n        if not isinstance(c[key], int) or c[key] <= 0:\n            raise ValueError(key + "는 양의 정수여야 합니다.")\n    if (\n        not isinstance(c["seeds"], list)\n        or not c["seeds"]\n        or any(not isinstance(value, int) or value < 0 for value in c["seeds"])\n        or len(set(c["seeds"])) != len(c["seeds"])\n    ):\n        raise ValueError("SEEDS는 서로 다른 0 이상의 정수 목록이어야 합니다.")\n    if c["mode"] == "baseline3" and len(c["seeds"]) != 3:\n        raise ValueError("baseline3는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "paper_suite" and len(c["seeds"]) != 3:\n        raise ValueError("paper_suite는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "supcon_compare" and c["seeds"] != [42]:\n        raise ValueError("supcon_compare의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] == "supcon_repeat" and c["seeds"] != [43, 44]:\n        raise ValueError("supcon_repeat의 확인 seed는 [43, 44]여야 합니다.")\n    if c["mode"] == "paper_screen" and c["seeds"] != [42]:\n        raise ValueError("paper_screen의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] in (\n        "sam_screen",\n        "final_candidate",\n        "web_skin_pmg_final",\n        "web_skin_medsiglip_linear",\n        "hair_medsiglip_linear",\n    ) and c["seeds"] != [42]:\n        raise ValueError(c["mode"] + "의 seed는 [42]여야 합니다.")\n    project = Path(c["project_root"])\n    if not project.is_dir():\n        raise FileNotFoundError("PROJECT_ROOT 폴더를 확인하세요: " + str(project))\n    if c["mode"] != "audit" and not (\n        c["audit_dir"] or c["expected_data_sha256"]\n    ):\n        raise ValueError("AUDIT_DIR 또는 확인된 EXPECTED_DATA_SHA256을 입력하세요.")\n    if c["data_zip"]:\n        candidates = [Path(c["data_zip"])]\n    else:\n        candidates = sorted(\n            p\n            for p in (project / "datasets").rglob(c["domain"] + "*")\n            if p.is_file() and zipfile.is_zipfile(p)\n        )\n    if len(candidates) != 1 or not zipfile.is_zipfile(candidates[0]):\n        raise ValueError(f"DATA_ZIP으로 ZIP 하나를 지정하세요: {candidates}")\n    source = candidates[0]\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]\n    local = Path(local_parent) / ("mediflow_" + run_id)\n    local.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(source) as archive:\n        required = (\n            source.stat().st_size + sum(m.file_size for m in archive.infolist()) + 2 * 1024**3\n        )\n    if required > shutil.disk_usage(local).free:\n        raise RuntimeError("Colab 압축 해제 공간이 부족합니다.")\n    copied = local / "input.zip"\n    shutil.copyfile(source, copied)\n    digest = engine.file_hash(copied)\n    if digest != engine.file_hash(source):\n        raise OSError("Drive ZIP 복사 내용 불일치")\n    classes = profiles[c["domain"]]\n    audit = None\n    if c["mode"] != "audit":\n        if c["audit_dir"]:\n            audit = verify_audit(c["audit_dir"], c["domain"], classes, digest)\n        else:\n            expected = c["expected_data_sha256"].strip().lower()\n            if len(expected) != 64 or any(ch not in "0123456789abcdef" for ch in expected):\n                raise ValueError("EXPECTED_DATA_SHA256은 64자리 SHA-256이어야 합니다.")\n            if digest != expected:\n                raise ValueError("DATA_ZIP이 확인된 SHA-256과 다릅니다.")\n            audit = {\n                "domain": c["domain"],\n                "class_names": classes,\n                "data_sha256": digest,\n                "protocol": "expected_sha256_v1",\n                "status": "independent_audit_skipped",\n                "limitations": [\n                    "Independent common audit was skipped by the project owner",\n                    "Person, lesion and capture-session leakage remains unverified",\n                    "Perceptual near-duplicate and clinical label checks were not performed",\n                ],\n            }\n    extract_zip(copied, local / "dataset")\n    settings = {\n        k: c[k]\n        for k in (\n            "domain",\n            "mode",\n            "seed",\n            "seeds",\n            "batch_size",\n            "epochs1",\n            "epochs2",\n            "extension_epochs",\n            "train_variant",\n        )\n    }\n    settings.update(\n        classes=classes,\n        data_sha256=digest,\n        source_hashes={k: signature(v) for k, v in sources.items()},\n        protocol="common_v1",\n        audit=signature(audit),\n        environment=dict(\n            tensorflow=tf.__version__,\n            keras=keras.__version__,\n            numpy=np.__version__,\n            python=platform.python_version(),\n        ),\n    )\n    if c["mode"] in (\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n        "web_skin_medsiglip_linear",\n        "hair_medsiglip_linear",\n    ):\n        settings["experiments"] = c.get("experiments", [])\n    if c["mode"] in ("sam_screen", "final_candidate", "web_skin_pmg_final"):\n        settings.update(\n            parent_run_dir=c.get("parent_run_dir"),\n            parent_model_sha256=c.get(\n                "parent_model_sha256", c.get("parent_stage1_sha256")\n            ),\n        )\n    if c["mode"] == "sam_screen":\n        settings["sam_rho"] = c.get("sam_rho")\n    if c["mode"] in ("web_skin_wsdan", "web_skin_paper_suite"):\n        settings.update(\n            attention_maps=c.get("attention_maps"),\n            crop_threshold=c.get("crop_threshold"),\n            drop_threshold=c.get("drop_threshold"),\n        )\n    if c["mode"] in ("web_skin_paper_suite", "web_skin_pmg_b1_384"):\n        settings.update(\n            pmg_jigsaw_grids=c.get("pmg_jigsaw_grids"),\n        )\n    if c["mode"] == "web_skin_paper_suite":\n        settings.update(\n            mixstyle_alpha=c.get("mixstyle_alpha"),\n            mixstyle_probability=c.get("mixstyle_probability"),\n        )\n    if c["mode"] in ("web_skin_medsiglip_linear", "hair_medsiglip_linear"):\n        settings.update(\n            model_id=c.get("model_id"),\n            embedding_shard_size=c.get("embedding_shard_size"),\n            linear_batch_size=c.get("linear_batch_size"),\n            linear_learning_rate=c.get("linear_learning_rate"),\n            linear_weight_decay=c.get("linear_weight_decay"),\n        )\n    sig = signature(settings)\n    if c["resume_dir"]:\n        output = Path(c["resume_dir"])\n        if c["mode"] == "audit":\n            raise ValueError("검사는 새 실행으로 시작하세요. RESUME_DIR을 비우세요.")\n        if engine.read_json(output / "run_config.json")["signature"] != sig:\n            raise ValueError(\n                "코드/설정/환경/데이터/검증이 다른 실행입니다. 새 결과 폴더를 사용하세요."\n            )\n    else:\n        output = project / "2_results" / c["domain"] / (c["mode"] + "_" + run_id)\n        output.mkdir(parents=True, exist_ok=False)\n        engine.write_json(\n            output / "run_config.json",\n            {\n                "signature": sig,\n                "settings": settings,\n                "code_commit_at_generation": commit,\n                "code_state": "embedded sources include uncommitted changes; exact sources saved",\n                "source_zip": str(source),\n                "audit_source": c["audit_dir"] or "expected_data_sha256_only",\n                "baseline": (\n                    (\n                        "Fixed PMG B0/256 Validation candidate reused; model not retrained"\n                        if c["mode"]\n                        in ("web_skin_pmg_final", "web_skin_medsiglip_linear")\n                        else (\n                            "Fixed Hair B1/384 Validation candidate reused; model not retrained"\n                            if c["mode"] == "hair_medsiglip_linear"\n                            else "Saved B0/256/CE Validation metrics reused; baseline not retrained"\n                        )\n                    )\n                    if c["mode"]\n                    in (\n                        "web_skin_wsdan",\n                        "web_skin_paper_suite",\n                        "web_skin_pmg_b1_384",\n                        "web_skin_pmg_final",\n                        "web_skin_medsiglip_linear",\n                        "hair_medsiglip_linear",\n                    )\n                    else "ImageNet pretrained EfficientNet; historical metrics not reused"\n                ),\n                "gpu": [str(d) for d in tf.config.list_physical_devices("GPU")],\n            },\n        )\n        for name, code in sources.items():\n            (output / (name + ".py")).write_text(code, encoding="utf-8")\n        engine.write_json(output / "class_names.json", classes)\n        if c["audit_dir"]:\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "audit_summary.json", output / "audit_summary.json"\n            )\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "image_inventory.csv", output / "image_inventory.csv"\n            )\n        elif audit:\n            engine.write_json(output / "audit_summary.json", audit)\n    context = dict(\n        config=c,\n        classes=classes,\n        signature=sig,\n        output=output,\n        local=local,\n        data_hash=digest,\n        audit=audit,\n        extracted=local / "dataset",\n        project=project,\n    )\n    if c["mode"] != "audit":\n        context["roots"] = roots_for(context["extracted"])\n        # Dataset ZIP is immutable and matches the audit; verify class folders again.\n        for root in context["roots"].values():\n            for split in ("train", "val", "test"):\n                found = sorted(p.name for p in (root / split).iterdir() if p.is_dir())\n                if found != sorted(classes):\n                    raise ValueError(f"클래스 불일치: {root / split}")\n    print("실행 결과:", output)\n    return context\n\n\ndef archive_results(context):\n    output = context["output"]\n    destination = output.parent / (output.name + "_results_" + uuid.uuid4().hex[:8] + ".zip")\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(output.rglob("*")):\n            if p.is_file() and p.suffix not in (".keras", ".tmp"):\n                archive.write(p, output.name + "/" + p.relative_to(output).as_posix())\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("보고서 ZIP 검사 실패")\n    print("로컬로 내려받을 결과 ZIP:", destination)\n    return destination\n\n\ndef audit_run(context):\n    from mediflow_datasets.common_audit import audit_dataset\n\n    summary = audit_dataset(\n        context["extracted"],\n        context["output"],\n        context["config"]["domain"],\n        context["classes"],\n        context["data_hash"],\n    )\n    engine.write_json(\n        context["output"] / "audit_manifest.json",\n        {\n            name: engine.file_hash(context["output"] / name)\n            for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv")\n        },\n    )\n    archive_results(context)\n    print("검사 상태:", summary["status"], "\\n학습 AUDIT_DIR:", context["output"])\n    return summary\n\n\ndef factory(context, variant, seed=None):\n    shuffle_seed = context["config"]["seed"] if seed is None else seed\n\n    def load(split, size, shuffle):\n        # All trials use exactly the same ORIGINAL validation and test images.\n        root = context["roots"][variant if split == "train" else "original"]\n        ds = keras.utils.image_dataset_from_directory(\n            root / split,\n            class_names=context["classes"],\n            label_mode="categorical",\n            image_size=(size, size),\n            interpolation="bilinear",\n            batch_size=context["config"]["batch_size"],\n            shuffle=shuffle,\n            seed=shuffle_seed if shuffle else None,\n        )\n        paths = [Path(p).relative_to(context["extracted"]).as_posix() for p in ds.file_paths]\n        return ds.prefetch(tf.data.AUTOTUNE), paths\n\n    return load\n\n\ndef run(context):\n    c, output = context["config"], context["output"]\n    if c["mode"] == "comparison":\n        trials = [\n            dict(id=kind, backbone="B0", size=224, loss="ce", variant=kind)\n            for kind in ("original", "augmented")\n        ]\n    else:\n        trials = [dict(spec, variant=c["train_variant"]) for spec in engine.TRIALS]\n    records = []\n    try:\n        for spec in trials:\n            spec["class_count"] = len(context["classes"])\n            spec["class_names"] = context["classes"]\n            load = factory(context, spec["variant"])\n            record = engine.run_trial(\n                spec,\n                load,\n                output,\n                context["signature"],\n                c["seed"],\n                c["epochs1"],\n                0 if c["mode"] == "comparison" else c["epochs2"],\n            )\n            records.append(record)\n            engine.write_json(output / "progress.json", {"completed": [r["id"] for r in records]})\n        if c["mode"] == "suite":\n            parent = records[-1]\n            records.append(\n                engine.extend_b1(\n                    parent,\n                    factory(context, c["train_variant"]),\n                    output,\n                    context["signature"],\n                    c["seed"],\n                    c["extension_epochs"],\n                )\n            )\n        engine.write_json(output / "all_validation_results.json", records)\n        return records\n    except Exception as exc:\n        engine.write_json(\n            output / ("failure_" + uuid.uuid4().hex[:8] + ".json"),\n            {\n                "error": repr(exc),\n                "completed": [r["id"] for r in records],\n                "resume_dir": str(output),\n            },\n        )\n        print("중단. 완료된 실험을 유지합니다. RESUME_DIR:", output)\n        raise\n\n\ndef confusion(ax, metrics, title):\n    cm = np.asarray(metrics["confusion_matrix"])\n    ax.imshow(cm, cmap="Blues")\n    codes = [f"C{i}" for i in range(len(cm))]\n    ax.set(\n        title=title,\n        xlabel="Predicted",\n        ylabel="True",\n        xticks=range(len(cm)),\n        yticks=range(len(cm)),\n        xticklabels=codes,\n        yticklabels=codes,\n    )\n    for i in range(len(cm)):\n        for j in range(len(cm)):\n            ax.text(\n                j,\n                i,\n                str(cm[i, j]),\n                ha="center",\n                va="center",\n                fontsize=8,\n                color="white" if cm[i, j] > cm.max() / 2 else "black",\n            )\n\n\ndef errors(context, csv_path, destination):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    from PIL import Image\n\n    frame = pd.read_csv(csv_path)\n    wrong = frame[frame.true_index != frame.pred_index].head(8)\n    fig, axes = plt.subplots(2, 4, figsize=(14, 7))\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, (_, row) in zip(axes.flat, wrong.iterrows(), strict=False):\n        path = (context["extracted"] / row["path"]).resolve()\n        if not path.is_relative_to(context["extracted"].resolve()):\n            raise ValueError("Prediction path escapes dataset")\n        with Image.open(path) as image:\n            ax.imshow(image.convert("RGB"))\n        ax.set_title(f"True C{row.true_index} / Pred C{row.pred_index}")\n    if wrong.empty:\n        fig.suptitle("No misclassifications")\n    fig.tight_layout()\n    fig.savefig(destination, dpi=160)\n    plt.close(fig)\n\n\ndef overview(context, records):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n\n    output = context["output"]\n    fig, axes = plt.subplots(\n        (len(records) + 1) // 2, 4, figsize=(24, 4.5 * ((len(records) + 1) // 2)), squeeze=False\n    )\n    for index, record in enumerate(records):\n        row, col = divmod(index, 2)\n        for offset, metric in enumerate(("accuracy", "loss")):\n            ax = axes[row, col * 2 + offset]\n            h = record["history"]\n            x = np.arange(1, len(h[metric]) + 1)\n            ax.plot(x, h[metric], label="Train")\n            ax.plot(x, h["val_" + metric], label="Validation")\n            if record["stage_boundary"] < len(x):\n                ax.axvline(record["stage_boundary"] + 0.5, ls="--", color="gray")\n            if "extension_boundary" in record:\n                ax.axvline(record["extension_boundary"] + 0.5, ls=":", color="green")\n            ax.set(title=record["id"] + " / " + metric, xlabel="Epoch", ylabel=metric)\n            if metric == "accuracy":\n                ax.set_ylim(0, 1)\n            ax.grid(alpha=0.25)\n            ax.legend()\n        directory = output / record["id"] / record["attempt"]\n        errors(\n            context, directory / "validation_predictions.csv", directory / "validation_errors.png"\n        )\n    fig.suptitle(context["config"]["domain"] + " / Loss definitions differ across CE, LS, Focal")\n    fig.tight_layout()\n    fig.savefig(output / "all_training_curves.png", dpi=180)\n    fig.savefig(output / "all_training_curves.pdf")\n    plt.show()\n    plt.close(fig)\n    table = pd.DataFrame(\n        [\n            dict(\n                experiment=r["id"],\n                selected_stage=r["selected_stage"],\n                validation_accuracy=r["validation"]["accuracy"],\n                validation_macro_f1=r["validation"]["macro_f1"],\n                parameters=r["parameters"],\n                seconds_this_trial=r["training_seconds"],\n                epochs=len(r["history"]["accuracy"]),\n                model_bytes=r["model_bytes"],\n            )\n            for r in records\n        ]\n    )\n    table.to_csv(output / "experiment_comparison.csv", index=False, encoding="utf-8-sig")\n    print(table.to_string(index=False))\n    fig, axes = plt.subplots(2, 1, figsize=(14, 11))\n    x = np.arange(len(records))\n    axes[0].bar(x - 0.2, table.validation_accuracy, 0.4, label="Validation Accuracy")\n    axes[0].bar(x + 0.2, table.validation_macro_f1, 0.4, label="Validation Macro F1")\n    axes[0].set(xticks=x, xticklabels=table.experiment, ylim=(0, 1))\n    axes[0].tick_params(axis="x", labelrotation=15)\n    axes[0].legend()\n    matrix = np.array([r["validation"]["class_f1"] for r in records])\n    axes[1].imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")\n    axes[1].set(\n        xticks=range(len(context["classes"])),\n        xticklabels=[f"C{i}" for i in range(len(context["classes"]))],\n        yticks=x,\n        yticklabels=table.experiment,\n        title="Validation class F1",\n    )\n    for i in range(len(records)):\n        for j in range(len(context["classes"])):\n            axes[1].text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=7)\n    fig.tight_layout()\n    fig.savefig(output / "validation_performance_dashboard.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    fig, axes = plt.subplots(\n        (len(records) + 2) // 3, 3, figsize=(18, 6 * ((len(records) + 2) // 3)), squeeze=False\n    )\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, r in zip(axes.flat, records, strict=False):\n        ax.axis("on")\n        confusion(ax, r["validation"], r["id"])\n    fig.tight_layout()\n    fig.savefig(output / "all_validation_confusion_matrices.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n\ndef finish(context, records):\n    import matplotlib.pyplot as plt\n\n    output = context["output"]\n    overview(context, records)\n    winner = engine.select_winner(records)\n    model_path = engine.selected_model_path(output, winner)\n    selection = dict(\n        winner=winner["id"],\n        model_sha256=engine.file_hash(model_path),\n        signature=context["signature"],\n        validation=winner["validation"],\n    )\n    selected_file = output / "selection_before_test.json"\n    if selected_file.exists() and engine.read_json(selected_file) != selection:\n        raise ValueError("이미 고정한 선택 모델이 다릅니다.")\n    if not selected_file.exists():\n        engine.write_json(selected_file, selection)\n    marker = output / "test_completed.json"\n    if marker.exists():\n        tested = engine.read_json(marker)\n        if tested["selection"] != selection:\n            raise ValueError("기존 Test 모델과 다릅니다.")\n        for name, digest in tested["hashes"].items():\n            if engine.file_hash(output / name) != digest:\n                raise ValueError("Test 파일이 변경됐습니다.")\n        metrics = tested["metrics"]\n    else:\n        keras.backend.clear_session()\n        model = keras.models.load_model(model_path, compile=False)\n        ds, paths = factory(context, winner["spec"]["variant"])(\n            "test", winner["spec"]["size"], False\n        )\n        metrics = engine.evaluate_to_files(model, ds, paths, output, "final_test")\n        engine.write_json(\n            marker,\n            dict(\n                selection=selection,\n                metrics=metrics,\n                hashes={\n                    name: engine.file_hash(output / name)\n                    for name in ("final_test_metrics.json", "final_test_predictions.csv")\n                },\n            ),\n        )\n        del model\n    fig, ax = plt.subplots(figsize=(8, 8))\n    confusion(ax, metrics, winner["id"] + " / Final Test")\n    fig.tight_layout()\n    fig.savefig(output / "final_test_confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    errors(context, output / "final_test_predictions.csv", output / "final_test_errors.png")\n    card = dict(\n        domain=context["config"]["domain"],\n        class_names=context["classes"],\n        normal_included="정상" in context["classes"],\n        validation=winner["validation"],\n        test=metrics,\n        selected_model=winner["id"],\n        model_sha256=selection["model_sha256"],\n        input_size=winner["spec"]["size"],\n        data_sha256=context["data_hash"],\n        status="public_data_candidate_not_device_validated",\n        limitations=context["audit"]["limitations"]\n        + [\n            "Single seed; small differences are not established as robust gains",\n            "Out-of-scope rejection absent; scores are not calibrated correctness",\n        ],\n    )\n    engine.write_json(output / "model_card.json", card)\n    package_parent = (\n        context["project"] / "2_results" / context["config"]["domain"] / "selected_models"\n    )\n    package = package_parent / (output.name + "_" + uuid.uuid4().hex[:8])\n    package.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(model_path, package / "model.keras")\n    if engine.file_hash(package / "model.keras") != selection["model_sha256"]:\n        raise OSError("모델 복사 불일치")\n    for name in (\n        "class_names.json",\n        "model_card.json",\n        "run_config.json",\n        "selection_before_test.json",\n        "audit_summary.json",\n        "final_test_metrics.json",\n        "common_engine.py",\n        "common_audit.py",\n        "common_workflow.py",\n    ):\n        shutil.copyfile(output / name, package / name)\n    engine.write_json(\n        package / "preprocessing.json",\n        dict(\n            input_shape=[winner["spec"]["size"], winner["spec"]["size"], 3],\n            color="RGB",\n            dtype="float32",\n            pixel_range=[0, 255],\n            external_normalization=False,\n            internal_rescaling="1/255",\n            resize="TensorFlow bilinear; no crop/pad; antialias=False",\n            exif_transpose=False,\n            output="softmax scores in class_names.json order",\n        ),\n    )\n    engine.write_json(\n        package / "manifest.json",\n        {p.name: engine.file_hash(p) for p in package.iterdir() if p.is_file()},\n    )\n    destination = Path(shutil.make_archive(str(package), "zip", package.parent, package.name))\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("모델 ZIP 손상")\n        manifest = engine.read_json(package / "manifest.json")\n        for name, digest in manifest.items():\n            if hashlib.sha256(archive.read(package.name + "/" + name)).hexdigest() != digest:\n                raise OSError("ZIP 내용 불일치: " + name)\n    destination.with_suffix(".zip.sha256").write_text(\n        engine.file_hash(destination), encoding="ascii"\n    )\n    archive_results(context)\n    print("선정 모델:", winner["id"], "\\nTest:", metrics, "\\n후보 ZIP:", destination)\n    return card\n', 'hair_medsiglip_linear': '"""Frozen MedSigLIP-448 linear-probe screening for Hair.\n\nOnly augmented Train and original Validation are read.  Test remains sealed until a\ncandidate is selected.  Image embeddings are written in deterministic shards so a\nColab interruption can resume without repeating completed feature extraction.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport platform\nimport time\nimport uuid\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom mediflow_datasets import common_engine as engine\n\nPROTOCOL = "hair_medsiglip_448_linear_probe_v1"\nTRIAL_ID = "hair_medsiglip_448_frozen_linear_seed_42"\nEXPERIMENTS = [TRIAL_ID]\nMODEL_ID = "google/medsiglip-448"\nEXPECTED_CLASSES = ["모낭사이홍반", "미세각질", "비듬", "탈모", "피지과다"]\nEXPECTED_COUNTS = {"train": 15047, "val": 1252}\nMODEL_CARD_URL = "https://huggingface.co/google/medsiglip-448"\nOFFICIAL_GUIDE_URL = (\n    "https://developers.google.com/health-ai-developer-foundations/medsiglip/get-started"\n)\nBASELINE = {\n    "id": "multires_b1_384_seed_42",\n    "validation_accuracy": 0.7963258785942492,\n    "validation_macro_f1": 0.7957146810115047,\n    "validation_count": 1252,\n    "data_sha256": "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156",\n    "source": "paper_screen_20260917_141641_874f03f1/multires_b1_384_seed_42",\n}\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}\n\n\ndef validate(context):\n    c = context["config"]\n    if c["domain"] != "hair" or c["mode"] != "hair_medsiglip_linear":\n        raise ValueError("Hair MedSigLIP 모드가 아닙니다.")\n    if context["classes"] != EXPECTED_CLASSES:\n        raise ValueError("Hair 클래스 순서가 기존 계약과 다릅니다.")\n    if context["data_hash"] != BASELINE["data_sha256"]:\n        raise ValueError("Hair clean ZIP SHA-256이 기존 실험과 다릅니다.")\n    if c["train_variant"] != "augmented":\n        raise ValueError("학습 데이터는 기존과 같은 augmented여야 합니다.")\n    if c["seed"] != 42 or c["seeds"] != [42]:\n        raise ValueError("선별 실험 seed는 42 하나로 고정합니다.")\n    if c.get("experiments") != EXPERIMENTS:\n        raise ValueError("MedSigLIP Linear 실험 하나만 실행합니다.")\n    fixed = {\n        "model_id": MODEL_ID,\n        "batch_size": 8,\n        "epochs1": 50,\n        "epochs2": 1,\n        "extension_epochs": 1,\n        "embedding_shard_size": 128,\n        "linear_batch_size": 256,\n        "linear_learning_rate": 1e-3,\n        "linear_weight_decay": 1e-4,\n    }\n    for key, expected in fixed.items():\n        if c.get(key) != expected:\n            raise ValueError(f"{key}는 {expected!r}로 고정합니다.")\n\n\ndef _images(context, split):\n    if split not in EXPECTED_COUNTS:\n        raise ValueError("이 실험은 Train과 Validation만 사용합니다.")\n    root = context["roots"]["augmented" if split == "train" else "original"] / split\n    paths, labels = [], []\n    for label, class_name in enumerate(context["classes"]):\n        folder = root / class_name\n        found = sorted(\n            path\n            for path in folder.rglob("*")\n            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS\n        )\n        paths.extend(found)\n        labels.extend([label] * len(found))\n    if len(paths) != EXPECTED_COUNTS[split]:\n        raise ValueError(f"{split} 이미지 수가 {EXPECTED_COUNTS[split]}장이 아닙니다: {len(paths)}")\n    return paths, np.asarray(labels, dtype=np.int64)\n\n\ndef _shard_paths(directory, split, index):\n    root = Path(directory) / "embedding_cache" / split\n    root.mkdir(parents=True, exist_ok=True)\n    return root / f"shard_{index:04d}.npz"\n\n\ndef _read_shard(path, expected_paths, expected_labels):\n    try:\n        with np.load(path, allow_pickle=False) as value:\n            embeddings = value["embeddings"]\n            labels = value["labels"]\n            paths = value["paths"].tolist()\n    except (OSError, ValueError, KeyError):\n        return None\n    if paths != expected_paths or not np.array_equal(labels, expected_labels):\n        return None\n    if (\n        embeddings.ndim != 2\n        or embeddings.shape[0] != len(paths)\n        or not np.isfinite(embeddings).all()\n    ):\n        return None\n    return embeddings.astype(np.float32, copy=False)\n\n\ndef _save_shard(path, embeddings, labels, paths):\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex[:8] + ".tmp.npz")\n    np.savez_compressed(\n        temporary,\n        embeddings=np.asarray(embeddings, dtype=np.float32),\n        labels=np.asarray(labels, dtype=np.int64),\n        paths=np.asarray(paths, dtype=np.str_),\n    )\n    temporary.replace(path)\n\n\ndef _load_encoder(token, model_id):\n    import torch\n    from transformers import AutoModel, AutoProcessor\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("MedSigLIP 특징 추출에는 Colab GPU가 필요합니다.")\n    processor = AutoProcessor.from_pretrained(model_id, token=token)\n    model = AutoModel.from_pretrained(\n        model_id,\n        token=token,\n        torch_dtype=torch.float16,\n        low_cpu_mem_usage=True,\n    ).eval().to("cuda")\n    if not hasattr(model, "get_image_features"):\n        raise TypeError("불러온 모델에 get_image_features가 없습니다.")\n    return model, processor\n\n\ndef _extract_split(context, split, model, processor):\n    import torch\n    from PIL import Image\n\n    paths, labels = _images(context, split)\n    relative = [path.relative_to(context["extracted"]).as_posix() for path in paths]\n    shard_size = context["config"]["embedding_shard_size"]\n    batch_size = context["config"]["batch_size"]\n    all_embeddings = []\n    for shard_index, start in enumerate(range(0, len(paths), shard_size)):\n        stop = min(start + shard_size, len(paths))\n        shard_path = _shard_paths(context["output"], split, shard_index)\n        cached = _read_shard(shard_path, relative[start:stop], labels[start:stop])\n        if cached is not None:\n            all_embeddings.append(cached)\n            print(f"{split} embedding cache: {stop} / {len(paths)}")\n            continue\n        pieces = []\n        for batch_start in range(start, stop, batch_size):\n            batch_stop = min(batch_start + batch_size, stop)\n            images = []\n            for path in paths[batch_start:batch_stop]:\n                with Image.open(path) as image:\n                    images.append(image.convert("RGB"))\n            inputs = processor(images=images, return_tensors="pt")\n            pixel_values = inputs["pixel_values"].to("cuda", dtype=torch.float16)\n            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):\n                features = model.get_image_features(pixel_values=pixel_values)\n                features = torch.nn.functional.normalize(features.float(), dim=-1)\n            pieces.append(features.cpu().numpy())\n        embeddings = np.concatenate(pieces).astype(np.float32, copy=False)\n        _save_shard(shard_path, embeddings, labels[start:stop], relative[start:stop])\n        all_embeddings.append(embeddings)\n        print(f"{split} embedding: {stop} / {len(paths)}")\n    return np.concatenate(all_embeddings), labels, relative\n\n\ndef _metrics(truth, probabilities):\n    return engine.classification_metrics(truth, probabilities, len(EXPECTED_CLASSES))\n\n\ndef _linear_probe(context, train_x, train_y, val_x, val_y):\n    import torch\n\n    torch.manual_seed(context["config"]["seed"])\n    torch.cuda.manual_seed_all(context["config"]["seed"])\n    device = "cuda"\n    head = torch.nn.Linear(train_x.shape[1], len(context["classes"])).to(device)\n    optimizer = torch.optim.AdamW(\n        head.parameters(),\n        lr=context["config"]["linear_learning_rate"],\n        weight_decay=context["config"]["linear_weight_decay"],\n    )\n    criterion = torch.nn.CrossEntropyLoss()\n    generator = torch.Generator().manual_seed(context["config"]["seed"])\n    dataset = torch.utils.data.TensorDataset(\n        torch.from_numpy(train_x), torch.from_numpy(train_y)\n    )\n    loader = torch.utils.data.DataLoader(\n        dataset,\n        batch_size=context["config"]["linear_batch_size"],\n        shuffle=True,\n        generator=generator,\n    )\n    val_tensor = torch.from_numpy(val_x).to(device)\n    best, history = None, []\n    for epoch in range(1, context["config"]["epochs1"] + 1):\n        head.train()\n        total_loss, total = 0.0, 0\n        for features, targets in loader:\n            features, targets = features.to(device), targets.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = criterion(head(features), targets)\n            loss.backward()\n            optimizer.step()\n            total_loss += float(loss.detach()) * len(targets)\n            total += len(targets)\n        head.eval()\n        with torch.inference_mode():\n            probabilities = torch.softmax(head(val_tensor), dim=1).cpu().numpy()\n        metrics = _metrics(val_y, probabilities)\n        row = {\n            "epoch": epoch,\n            "train_loss": total_loss / total,\n            "val_accuracy": metrics["accuracy"],\n            "val_macro_f1": metrics["macro_f1"],\n        }\n        history.append(row)\n        key = (metrics["accuracy"], metrics["macro_f1"], -epoch)\n        if best is None or key > best["key"]:\n            best = {\n                "key": key,\n                "epoch": epoch,\n                "state": {name: value.detach().cpu() for name, value in head.state_dict().items()},\n                "probabilities": probabilities,\n                "metrics": metrics,\n            }\n        print(\n            f"Linear epoch {epoch:02d}/50 - loss {row[\'train_loss\']:.4f} - "\n            f"val_acc {row[\'val_accuracy\']:.4f} - val_macro_f1 {row[\'val_macro_f1\']:.4f}"\n        )\n    return best, history\n\n\ndef _save_predictions(path, paths, truth, probabilities):\n    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["path", "true_index", "pred_index", *[f"prob_C{i}" for i in range(5)]])\n        for name, target, probs in zip(paths, truth, probabilities, strict=True):\n            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])\n\n\ndef run(context, hf_token):\n    validate(context)\n    if not hf_token or not hf_token.startswith("hf_"):\n        raise ValueError("Colab Secrets의 HF_TOKEN 읽기 토큰을 확인하세요.")\n    output = context["output"]\n    completed = output / "medsiglip_linear_completed.json"\n    if completed.exists():\n        record = engine.read_json(completed)\n        if record["signature"] != context["signature"]:\n            raise ValueError("완료 기록의 설정이 현재 실행과 다릅니다.")\n        for relative, digest in record["artifact_hashes"].items():\n            if engine.file_hash(output / relative) != digest:\n                raise ValueError("완료 산출물이 변경됐습니다: " + relative)\n        return record\n    started = time.monotonic()\n    model, processor = _load_encoder(hf_token, context["config"]["model_id"])\n    train_x, train_y, _ = _extract_split(context, "train", model, processor)\n    val_x, val_y, val_paths = _extract_split(context, "val", model, processor)\n    del model, processor\n    import torch\n\n    torch.cuda.empty_cache()\n    best, history = _linear_probe(context, train_x, train_y, val_x, val_y)\n    head_path = output / "medsiglip_linear_head.pt"\n    torch.save(\n        {\n            "state_dict": best["state"],\n            "input_dim": int(train_x.shape[1]),\n            "class_names": context["classes"],\n            "model_id": MODEL_ID,\n            "protocol": PROTOCOL,\n        },\n        head_path,\n    )\n    engine.write_json(output / "validation_metrics.json", best["metrics"])\n    engine.write_json(output / "linear_history.json", history)\n    _save_predictions(\n        output / "validation_predictions.csv", val_paths, val_y, best["probabilities"]\n    )\n    environment = {\n        "python": platform.python_version(),\n        "numpy": np.__version__,\n        "torch": torch.__version__,\n        "torch_cuda": torch.version.cuda,\n        "gpu": torch.cuda.get_device_name(0),\n    }\n    try:\n        import transformers\n\n        environment["transformers"] = transformers.__version__\n    except ImportError:\n        pass\n    engine.write_json(output / "medsiglip_environment.json", environment)\n    artifacts = [\n        "medsiglip_linear_head.pt",\n        "validation_metrics.json",\n        "linear_history.json",\n        "validation_predictions.csv",\n        "medsiglip_environment.json",\n    ]\n    record = {\n        "id": TRIAL_ID,\n        "protocol": PROTOCOL,\n        "signature": context["signature"],\n        "model_id": MODEL_ID,\n        "encoder_frozen": True,\n        "head": "single Linear layer",\n        "best_epoch": best["epoch"],\n        "validation": best["metrics"],\n        "elapsed_seconds": time.monotonic() - started,\n        "test_evaluated": False,\n        "artifact_hashes": {name: engine.file_hash(output / name) for name in artifacts},\n    }\n    engine.write_json(completed, record)\n    return record\n\n\ndef _selected(record):\n    new = record["validation"]\n    return (\n        new["macro_f1"] > BASELINE["validation_macro_f1"]\n        or (\n            new["macro_f1"] == BASELINE["validation_macro_f1"]\n            and new["accuracy"] > BASELINE["validation_accuracy"]\n        )\n    )\n\n\ndef _plots(context, record):\n    import matplotlib.pyplot as plt\n\n    output = context["output"]\n    history = engine.read_json(output / "linear_history.json")\n    figure, axes = plt.subplots(1, 2, figsize=(13, 5))\n    epochs = [row["epoch"] for row in history]\n    axes[0].plot(epochs, [row["train_loss"] for row in history], label="Train loss")\n    axes[0].set(title="MedSigLIP Linear Head Loss", xlabel="Epoch", ylabel="Loss")\n    axes[1].plot(epochs, [row["val_accuracy"] for row in history], label="Validation Accuracy")\n    axes[1].plot(epochs, [row["val_macro_f1"] for row in history], label="Validation Macro F1")\n    axes[1].set(title="Validation Metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1))\n    for axis in axes:\n        axis.grid(alpha=0.3)\n        axis.legend()\n    figure.tight_layout()\n    figure.savefig(output / "medsiglip_training_curves.png", dpi=180)\n    plt.close(figure)\n\n    matrix = np.asarray(record["validation"]["confusion_matrix"])\n    figure, axis = plt.subplots(figsize=(7, 6))\n    image = axis.imshow(matrix, cmap="Blues")\n    for row in range(5):\n        for col in range(5):\n            axis.text(col, row, str(matrix[row, col]), ha="center", va="center")\n    axis.set_xticks(range(5), context["classes"], rotation=35, ha="right")\n    axis.set_yticks(range(5), context["classes"])\n    axis.set(xlabel="Predicted", ylabel="True", title="MedSigLIP Validation Confusion Matrix")\n    figure.colorbar(image, ax=axis)\n    figure.tight_layout()\n    figure.savefig(output / "medsiglip_validation_confusion_matrix.png", dpi=180)\n    plt.close(figure)\n\n    labels = ["Hair B1·384 v2", "MedSigLIP-448 Linear"]\n    acc = [BASELINE["validation_accuracy"], record["validation"]["accuracy"]]\n    f1 = [BASELINE["validation_macro_f1"], record["validation"]["macro_f1"]]\n    x = np.arange(2)\n    figure, axis = plt.subplots(figsize=(9, 5))\n    axis.bar(x - 0.18, acc, 0.36, label="Validation Accuracy")\n    axis.bar(x + 0.18, f1, 0.36, label="Validation Macro F1")\n    axis.set_xticks(x, labels)\n    axis.set_ylim(max(0, min(acc + f1) - 0.08), 1)\n    axis.set_title("Hair Validation Comparison")\n    axis.grid(axis="y", alpha=0.3)\n    axis.legend()\n    for index, value in enumerate(acc):\n        axis.text(index - 0.18, value + 0.006, f"{value:.4f}", ha="center")\n    for index, value in enumerate(f1):\n        axis.text(index + 0.18, value + 0.006, f"{value:.4f}", ha="center")\n    figure.tight_layout()\n    figure.savefig(output / "medsiglip_vs_hair_v2_validation.png", dpi=180)\n    plt.close(figure)\n\n\ndef _report_archive(context):\n    output = context["output"]\n    destination = output.parent / (output.name + "_reports_" + uuid.uuid4().hex[:8] + ".zip")\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob("*")):\n            if not path.is_file() or "embedding_cache" in path.parts:\n                continue\n            archive.write(path, output.name + "/" + path.relative_to(output).as_posix())\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("보고서 ZIP 검사 실패")\n    return destination\n\n\ndef summarize(context, record):\n    validate(context)\n    _plots(context, record)\n    eligible = _selected(record)\n    summary = {\n        "protocol": PROTOCOL,\n        "hypothesis": "Frozen medical image embeddings improve Hair linear separability.",\n        "fixed": {\n            "data_sha256": context["data_hash"],\n            "class_names": context["classes"],\n            "train_variant": "augmented",\n            "train_count": EXPECTED_COUNTS["train"],\n            "validation_count": EXPECTED_COUNTS["val"],\n            "seed": 42,\n        },\n        "changed_variable": "Frozen MedSigLIP-448 image encoder plus a single Linear head",\n        "necessary_accompanying_change": (\n            "Input preprocessing follows the official MedSigLIP processor at 448 pixels."\n        ),\n        "baseline": BASELINE,\n        "medsiglip_validation": record["validation"],\n        "selection_rule": "Higher Validation Macro F1; Accuracy breaks an exact Macro F1 tie.",\n        "eligible_for_separate_final_test": eligible,\n        "hair_v2_replaced": False,\n        "test_evaluated": False,\n        "model_card": MODEL_CARD_URL,\n        "official_guide": OFFICIAL_GUIDE_URL,\n    }\n    engine.write_json(context["output"] / "medsiglip_validation_summary.json", summary)\n    archive = _report_archive(context)\n    print("Validation 비교 완료. Test는 실행하지 않았습니다.")\n    print("결과 ZIP:", archive)\n    return summary, archive\n'}
BUILD_COMMIT = '5997df7e2d4e3ea2b26417f4725467140e9d6f64'
PROFILES = {'hair': ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']}
package = types.ModuleType('mediflow_datasets')
package.__path__ = []
sys.modules['mediflow_datasets'] = package
for name, source in SOURCES.items():
    module = types.ModuleType('mediflow_datasets.' + name)
    sys.modules[module.__name__] = module
    exec(compile(source, name + '.py', 'exec'), module.__dict__)
from mediflow_datasets.common_workflow import prepare
from mediflow_datasets.hair_medsiglip_linear import run, summarize


## 4. 데이터 준비와 실행 폴더 생성

ZIP 해시와 클래스 순서를 확인하고 로컬 런타임에 압축을 풉니다. 처음 실행한 뒤 출력되는 결과
폴더를 메모해 두세요. 중단 시 그 전체 경로를 위 `RESUME_DIR`에 입력합니다.


In [ ]:
config = dict(
    domain=DOMAIN, project_root=PROJECT_ROOT, data_zip=DATA_ZIP,
    audit_dir=AUDIT_DIR, expected_data_sha256=EXPECTED_DATA_SHA256,
    resume_dir=RESUME_DIR, mode=MODE, seed=SEED, seeds=SEEDS,
    batch_size=BATCH_SIZE, epochs1=LINEAR_EPOCHS, epochs2=1,
    extension_epochs=1, train_variant=TRAIN_VARIANT,
    experiments=RUN_EXPERIMENTS, model_id=MODEL_ID,
    embedding_shard_size=EMBEDDING_SHARD_SIZE,
    linear_batch_size=LINEAR_BATCH_SIZE,
    linear_learning_rate=LINEAR_LEARNING_RATE,
    linear_weight_decay=LINEAR_WEIGHT_DECAY,
)
context = prepare(config, PROFILES, SOURCES, BUILD_COMMIT)
print('결과 폴더 / 중단 시 RESUME_DIR:', context['output'])
print('데이터 SHA-256:', context['data_hash'])
print('클래스 순서:', context['classes'])


## 5. MedSigLIP 특징 추출과 Linear 분류기 학습

처음에는 MedSigLIP 모델을 내려받고 16,299장의 특징을 추출하므로 Web Skin보다 오래 걸립니다.
`train embedding: 128 / 15047` 같은 출력은 완료된 사진 수입니다. 캐시가 있으면 다음 실행에서
`embedding cache`로 표시됩니다. 토큰은 모델 다운로드에만 사용하며 결과에 저장하지 않습니다.


In [ ]:
record = run(context, HF_TOKEN)
record['validation']


## 6. 기존 v2와 Validation 비교 자료 생성

학습곡선, 혼동행렬, 기존 Hair B1·384 v2와의 Accuracy·Macro F1 비교 그림과 보고서 ZIP을 만듭니다.
선정 규칙은 Validation Macro F1 우선, 정확히 같으면 Accuracy입니다. 이 실행만으로 현재 v2를
교체하거나 Test를 평가하지 않습니다. embedding 캐시는 재개용이므로 보고서 ZIP에서 제외됩니다.


In [ ]:
summary, report_zip = summarize(context, record)
summary
